# 06 — Generate evaluation outputs and pairwise files (Task 6)

Accelerator: **GPU T4 x2**. **Add Input** → saved outputs of notebooks 02 (SFT), 03 (DPO-HH), 04 (DPO-UF) and
05 (LoRA). Uses identical decoding for both sides of every comparison. Two models are generated at a time,
one per GPU.

| Comparison | Eval set | Candidate | Baseline |
|---|---|---|---|
| UltraFeedback DPO | `eval_sets/alpaca_eval_300_seed42.json` | DPO-UF best | SFT best |
| HH-RLHF DPO | `eval_sets/hh_rlhf_eval_300_seed42.json` | DPO-HH best | SFT best |
| LoRA persona | `eval_sets/persona_style_eval.json` | SmolLM2 + LoRA | SmolLM2-360M-Instruct |

In [ ]:
# Clone (or update) the project repo into /kaggle/working. Notebook settings: Internet ON.
import os
REPO_URL = "https://github.com/mahamasiff/cs5326-PA2-27100015.git"
REPO_DIR = "/kaggle/working/cs5326-PA2-27100015"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
!git log --oneline -1

In [ ]:
import subprocess
def sh(cmd):
    """Run a shell command, stream its output, and stop the notebook if it fails."""
    print("$", cmd, flush=True)
    result = subprocess.run(cmd, shell=True)
    if result.returncode != 0:
        raise RuntimeError(f"command failed with exit code {result.returncode}: {cmd}")

In [ ]:
# Manual section 5.2 plus PEFT/TRL for the LoRA task. torchao is removed to avoid the PEFT conflict.
!python -m pip uninstall -y -q torchao || true
!python -m pip install -q "transformers>=4.43.0,<5" "datasets>=2.18,<5" huggingface_hub accelerate tqdm pyyaml sentencepiece safetensors pytest "peft>=0.11.0" "trl>=0.24,<0.30"
!python -m pip install -q -e . --no-deps

In [ ]:
# Locate artifacts from earlier notebooks attached under /kaggle/input (Add Input -> your notebook output).
import glob, os
def find_input(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"Attach the notebook output that contains {pattern}"
    print(pattern, "->", hits[0])
    return hits[0]

In [ ]:
SFT = os.path.dirname(find_input("sft/best/checkpoint_info.json"))
DPO_HH = os.path.dirname(find_input("dpo_hh/best/checkpoint_info.json"))
DPO_UF = os.path.dirname(find_input("dpo_uf/best/checkpoint_info.json"))
LORA = os.path.dirname(find_input("lora_dpo/final/adapter_config.json"))
SMOL = "HuggingFaceTB/SmolLM2-360M-Instruct"
ALPACA, HH, PERSONA = "eval_sets/alpaca_eval_300_seed42.json", "eval_sets/hh_rlhf_eval_300_seed42.json", "eval_sets/persona_style_eval.json"
G = "outputs/generations"

In [ ]:
DECODE = "--max_new_tokens 512 --batch_size 16 --repetition_penalty 1.0"  # keep identical across notebooks 01 and 06

In [ ]:
# Sanity check on 3 prompts per checkpoint (manual section 12): path, tokenizer, prompt format, EOS, length.
!python scripts/generate.py --model_path {SFT} --generator sft --eval_set {ALPACA} --output {G}/sanity_sft.json --limit 3 {DECODE}
!python scripts/generate.py --model_path {DPO_UF} --generator dpo_uf --eval_set {ALPACA} --output {G}/sanity_dpo_uf.json --limit 3 {DECODE}
!python scripts/generate.py --model_path {DPO_HH} --generator dpo_hh --eval_set {HH} --output {G}/sanity_dpo_hh.json --limit 3 {DECODE}
!python scripts/generate.py --model_path {SMOL} --adapter_path {LORA} --prompt_format chat_template --generator smollm2-lora-dpo --eval_set {PERSONA} --output {G}/sanity_lora.json --limit 3 --max_new_tokens 256

Inspect the sanity outputs above. Only continue once the format, EOS handling, and length look reasonable.

In [ ]:
# AlpacaEval: SFT on GPU 0 and DPO-UF on GPU 1 in parallel.
sh(f"python scripts/generate.py --model_path {SFT} --generator sft --eval_set {ALPACA} --output {G}/alpaca_sft.json --device cuda:0 {DECODE} > {G}/log_alpaca_sft.txt 2>&1 & "
   f"python scripts/generate.py --model_path {DPO_UF} --generator dpo_uf --eval_set {ALPACA} --output {G}/alpaca_dpo_uf.json --device cuda:1 {DECODE} > {G}/log_alpaca_dpo_uf.txt 2>&1 & wait")
!tail -n 2 {G}/log_alpaca_*.txt

In [ ]:
# HH helpful/harmless set: SFT and DPO-HH in parallel.
sh(f"python scripts/generate.py --model_path {SFT} --generator sft --eval_set {HH} --output {G}/hh_sft.json --device cuda:0 {DECODE} > {G}/log_hh_sft.txt 2>&1 & "
   f"python scripts/generate.py --model_path {DPO_HH} --generator dpo_hh --eval_set {HH} --output {G}/hh_dpo_hh.json --device cuda:1 {DECODE} > {G}/log_hh_dpo_hh.txt 2>&1 & wait")
!tail -n 2 {G}/log_hh_*.txt

In [ ]:
# Persona set: SmolLM2 base vs SmolLM2 + LoRA (its own chat template, same decoding for both).
PERSONA_DECODE = "--max_new_tokens 256 --batch_size 16 --repetition_penalty 1.0"
sh(f"python scripts/generate.py --model_path {SMOL} --prompt_format chat_template --generator smollm2-instruct --eval_set {PERSONA} --output {G}/persona_base.json --device cuda:0 {PERSONA_DECODE} > {G}/log_persona_base.txt 2>&1 & "
   f"python scripts/generate.py --model_path {SMOL} --adapter_path {LORA} --prompt_format chat_template --generator smollm2-lora-dpo --eval_set {PERSONA} --output {G}/persona_lora.json --device cuda:1 {PERSONA_DECODE} > {G}/log_persona_lora.txt 2>&1 & wait")
!tail -n 2 {G}/log_persona_*.txt

In [ ]:
# Parallel runs report success even if one process failed, so verify every file is complete.
import json
expected = {"alpaca_sft": ALPACA, "alpaca_dpo_uf": ALPACA, "hh_sft": HH, "hh_dpo_hh": HH,
            "persona_base": PERSONA, "persona_lora": PERSONA}
for name, eval_set in expected.items():
    path = f"{G}/{name}.json"
    assert os.path.exists(path), f"{path} missing: check {G}/log_{name}.txt"
    g, n = json.load(open(path)), len(json.load(open(eval_set)))
    assert len(g) == n, f"{path} has {len(g)}/{n} outputs: re-run its cell (it resumes) and check the log"
    print(f"{name}: {len(g)} outputs, {sum(x['hit_max_new_tokens'] for x in g)} hit max_new_tokens, "
          f"mean chars {sum(len(x['output']) for x in g) / len(g):.0f}")

In [ ]:
# Blinded pairwise files (A/B randomized with seed 42; identities only in "hidden").
!python scripts/build_pairs.py --candidate {G}/alpaca_dpo_uf.json --candidate_name dpo_uf --baseline {G}/alpaca_sft.json --baseline_name sft --output outputs/eval/dpo_uf_vs_sft_pairs.jsonl
!python scripts/build_pairs.py --candidate {G}/hh_dpo_hh.json --candidate_name dpo_hh --baseline {G}/hh_sft.json --baseline_name sft --output outputs/eval/dpo_hh_vs_sft_pairs.jsonl
!python scripts/build_pairs.py --candidate {G}/persona_lora.json --candidate_name lora_dpo --baseline {G}/persona_base.json --baseline_name smollm2_instruct --output outputs/eval/lora_vs_base_pairs.jsonl

In [ ]:
!cd outputs && zip -q -r /kaggle/working/generation_outputs.zip generations eval
!ls -lh /kaggle/working/generation_outputs.zip

**Save Version** so the generations and pairwise files can be attached to the judging notebook.